In [ ]:
# !conda create -n skmob python=3.9.0
# !conda install conda-forge scikit-mobility

In [1]:
import networkx as nx
import pandas as pd # version 2.3.1
import geopandas as gpd #1.0.1
import numpy as np # 2.2.6
import shapely   # 2.0.7 , measure importance of features through SHAP values
import skmob

Cannot find header.dxf (GDAL_DATA is not defined)


In [ ]:
import os

data_folder = "data/2017-2021"
# datasets is aggregated commuter trajectories
datasets = {}

for filename in os.listdir(data_folder):
    # filename.split("_") creates list of items split from the filename -> [-1] access last item -> .split(".") splits last item by "."-> [0] accesses the first item which is the state code
    state_code = filename.split("_")[-1].split(".")[0]
    file_path = os.path.join(data_folder, filename)
    df = pd.read_csv(file_path)
    # Add the DataFrame to the container with the state code as the label
    datasets[state_code] = df
    

In [33]:
datasets['FL'].groupby("Otract").nunique()

,Dtract,EST
Otract,,
12001000201,24,15
12001000202,28,15
12001000301,27,19
12001000302,27,13
12001000400,28,16
...,...,...
12133970104,14,10
12133970200,27,10
12133970301,25,12


In [34]:
datasets['FL'].groupby("Dtract").nunique()

,Otract,EST
Dtract,,
1003010100,1,1
1003010200,1,1
1003010400,5,4
1003010500,5,4
1003010704,2,1
...,...,...
55139001700,1,1
55141010400,1,1
55141011100,1,1


In [36]:
datasets['FL']['Otract'].unique()

array([12001000201, 12001000202, 12001000301, ..., 12133970301,
       12133970302, 12133970303])

Some census tracts are only included in the dataset as destination tracts. These cencus tracts have commuters entering them for work, but no recorded population leaving them to work elsewhere. So these census tracts should be recorded as having a population of 0. 

In [22]:
# population of census tract is the sum of travelers when grouped by an Otract id matching the census tract id
# number of jobs for census tract is the sum of travelers when grouped by a Dtract id matching the census tract id
grouped_dsets = {}

In [ ]:
for (state_code, df) in enumerate(datasets):
    grouped = pd.DataFrame()
    grouped['Id'] = df['Otract'].unique()
    population = df.groupby('Otract')['EST'].sum()
    grouped['Population'] = grouped['Id'].map(population)
    
    # Add census tract ids that are within Dtract but not Otract
    # Find unique Dtract ids absent in Otract
    dtract_only = set(df['Dtract'].unique()) - set(df['Otract'].unique())
    # Create rows for these ids with Population set to NA
    dtract_only = pd.DataFrame([[{id, np.nan} for id in dtract_only]], columns = ["Id", "Population"])
    grouped = pd.concat([grouped, dtract_only])
    # Now calculate the number of jobs for each census tract. There may be census tracts that serve as origins and not destinations, and vice
    # versa, so calculate the jobs from both Otract and Dtract.
    jobs = df.groupby('Dtract')['EST'].sum()
    grouped['Number of Jobs'] = grouped['Id'].map(jobs)
    grouped.loc("Number of Jobs") = df.groupby('Otract')['EST'].sum()

    grouped_dsets[state_code] = grouped

In [32]:
datasets['FL']

,Otract,Dtract,EST
0,12001000201,12001000201,405.0
1,12001000201,12001000202,40.0
2,12001000201,12001000301,20.0
3,12001000201,12001000400,225.0
4,12001000201,12001000500,35.0
...,...,...,...
220720,12133970303,12133970103,55.0
220721,12133970303,12133970104,105.0
220722,12133970303,12133970200,15.0
220723,12133970303,12133970302,50.0


In [ ]:
assert(datasets['AK'])